# 🏋️ Intel Image Classifier — Colab Training Notebook

**Adımlar:**
1. GPU aktifleştir
2. GitHub repo clone
3. Kaggle'dan veri seti indir
4. Custom CNN eğit
5. ResNet18 (Transfer Learning) eğit
6. Karşılaştırma
7. Optuna HPO
8. ONNX Export
9. Dosyaları indir

## 1. GPU Kontrolü
**Runtime > Change runtime type > GPU (T4)** seç.

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    print("⚠️ GPU bulunamadı!")
print(f"Cihaz: {device}")

## 2. Projeyi Clone Et

In [ ]:
REPO_URL = "https://github.com/KULLANICI_ADIN/intel-image-classifier.git"  # <-- DEĞİŞTİR

!git clone {REPO_URL}
%cd intel-image-classifier
!pip install -q optuna

## 3. Kaggle'dan Veri Seti İndir

In [ ]:
!pip install -q kaggle
from google.colab import files
print("kaggle.json dosyanı yükle:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d puneet6060/intel-image-classification -p data/
!unzip -q data/intel-image-classification.zip -d data/
print("\n✅ Veri seti indirildi!")
!ls data/

In [ ]:
import os
TRAIN_DIR = "data/seg_train/seg_train"
TEST_DIR = "data/seg_test/seg_test"
print(f"Train: {os.path.exists(TRAIN_DIR)} | Test: {os.path.exists(TEST_DIR)}")
if os.path.exists(TRAIN_DIR):
    print(f"Sınıflar: {os.listdir(TRAIN_DIR)}")

## 4. Veri Pipeline

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data.transforms import get_train_transforms, get_test_transforms
from src.data.dataset import load_dataset, get_class_distribution, CLASS_NAMES
from src.data.dataloader import create_dataloaders

train_dataset = load_dataset(TRAIN_DIR, transform=get_train_transforms())
test_dataset = load_dataset(TEST_DIR, transform=get_test_transforms())

dist = get_class_distribution(train_dataset)
print(f"\nSınıf dağılımı: {dist}")

In [ ]:
BATCH_SIZE = 32
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset, test_dataset, batch_size=BATCH_SIZE, val_split=0.2, num_workers=2
)

images, labels = next(iter(train_loader))
print(f"Batch: {images.shape} | Labels: {labels.shape}")

## 5. Custom CNN Eğitimi

In [ ]:
import torch.nn as nn
import time
from src.models.custom_cnn import CustomCNN
from src.training.optimizer_factory import create_optimizer, create_scheduler
from src.training.trainer import Trainer

custom_model = CustomCNN(num_classes=6, num_blocks=4, base_filters=32, dropout_rate=0.3)
print(f"Parametre sayısı: {custom_model.count_parameters()}")

criterion = nn.CrossEntropyLoss()
optimizer = create_optimizer(custom_model, 'Adam', learning_rate=1e-3)
scheduler = create_scheduler(optimizer, step_size=5, gamma=0.5)

trainer = Trainer(custom_model, criterion, optimizer, device, scheduler, 'models')

t0 = time.time()
custom_history = trainer.fit(train_loader, val_loader, num_epochs=20, early_stopping_patience=5, model_name='custom_cnn')
custom_time = time.time() - t0
print(f"Süre: {custom_time:.1f}s")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(custom_history['train_loss'], label='Train', color='#6366f1', lw=2)
ax1.plot(custom_history['val_loss'], label='Val', color='#f43f5e', lw=2)
ax1.set_title('Custom CNN — Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(custom_history['train_acc'], label='Train', color='#6366f1', lw=2)
ax2.plot(custom_history['val_acc'], label='Val', color='#10b981', lw=2)
ax2.set_title('Custom CNN — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"En iyi Val Accuracy: {max(custom_history['val_acc']):.4f}")

## 6. ResNet18 Transfer Learning Eğitimi

In [ ]:
from src.models.transfer_model import TransferResNet18

resnet_model = TransferResNet18(num_classes=6, dropout_rate=0.3, freeze_backbone=True, unfreeze_last_n=1)
print(f"Parametre sayısı: {resnet_model.count_parameters()}")

criterion = nn.CrossEntropyLoss()
optimizer = create_optimizer(resnet_model, 'Adam', learning_rate=5e-4)
scheduler = create_scheduler(optimizer, step_size=5, gamma=0.5)

trainer_resnet = Trainer(resnet_model, criterion, optimizer, device, scheduler, 'models')

t0 = time.time()
resnet_history = trainer_resnet.fit(train_loader, val_loader, num_epochs=15, early_stopping_patience=5, model_name='resnet18')
resnet_time = time.time() - t0
print(f"Süre: {resnet_time:.1f}s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(resnet_history['train_loss'], label='Train', color='#6366f1', lw=2)
ax1.plot(resnet_history['val_loss'], label='Val', color='#f43f5e', lw=2)
ax1.set_title('ResNet18 — Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(resnet_history['train_acc'], label='Train', color='#6366f1', lw=2)
ax2.plot(resnet_history['val_acc'], label='Val', color='#10b981', lw=2)
ax2.set_title('ResNet18 — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"En iyi Val Accuracy: {max(resnet_history['val_acc']):.4f}")

## 7. Karşılaştırma

In [ ]:
from src.evaluation.metrics import calculate_f1, classification_summary, compute_confusion_matrix
import seaborn as sns

trainer.load_best_model('custom_cnn')
trainer_resnet.load_best_model('resnet18')

f1_custom = calculate_f1(custom_model, test_loader, device)
f1_resnet = calculate_f1(resnet_model, test_loader, device)

print('=' * 50)
print(f'Custom CNN  — Val Acc: {max(custom_history["val_acc"]):.4f} | F1: {f1_custom:.4f}')
print(f'ResNet18    — Val Acc: {max(resnet_history["val_acc"]):.4f} | F1: {f1_resnet:.4f}')
print('=' * 50)

print('\n--- Custom CNN ---')
print(classification_summary(custom_model, test_loader, device, CLASS_NAMES))
print('\n--- ResNet18 ---')
print(classification_summary(resnet_model, test_loader, device, CLASS_NAMES))

In [ ]:
cm_custom = compute_confusion_matrix(custom_model, test_loader, device)
cm_resnet = compute_confusion_matrix(resnet_model, test_loader, device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title('Custom CNN', fontweight='bold'); ax1.set_xlabel('Tahmin'); ax1.set_ylabel('Gerçek')

sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Greens', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
ax2.set_title('ResNet18', fontweight='bold'); ax2.set_xlabel('Tahmin'); ax2.set_ylabel('Gerçek')
plt.tight_layout(); plt.show()

## 8. Sonuçları JSON'a Kaydet
Streamlit Model Comparison sayfası bu dosyayı okuyacak.

In [ ]:
import json

results = {
    'custom_cnn': {
        'val_accuracy': max(custom_history['val_acc']),
        'val_f1_score': f1_custom,
        'train_loss': custom_history['train_loss'][-1],
        'val_loss': min(custom_history['val_loss']),
        'training_time_sec': custom_time,
        'total_params': custom_model.count_parameters()['total'],
        'trainable_params': custom_model.count_parameters()['trainable'],
        'history': custom_history
    },
    'resnet18': {
        'val_accuracy': max(resnet_history['val_acc']),
        'val_f1_score': f1_resnet,
        'train_loss': resnet_history['train_loss'][-1],
        'val_loss': min(resnet_history['val_loss']),
        'training_time_sec': resnet_time,
        'total_params': resnet_model.count_parameters()['total'],
        'trainable_params': resnet_model.count_parameters()['trainable'],
        'history': resnet_history
    }
}

with open('models/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('✅ Sonuçlar models/results.json dosyasına kaydedildi!')

## 9. Optuna HPO (Opsiyonel)
GPU zamanı yer, dikkatli kullan.

In [ ]:
from src.training.hpo import run_hpo_study

hpo_results = run_hpo_study(
    train_loader=train_loader, val_loader=val_loader,
    model_type='custom_cnn', device=device,
    n_trials=10, num_epochs=10
)
print(f"\nEn iyi parametreler: {hpo_results['best_params']}")
print(f"En iyi accuracy: {hpo_results['best_value']:.4f}")

In [ ]:
# En iyi parametrelerle final eğitim
best_p = hpo_results['best_params']
final_model = CustomCNN(num_classes=6, num_blocks=best_p.get('num_blocks', 4),
                        base_filters=best_p.get('base_filters', 32),
                        dropout_rate=best_p.get('dropout_rate', 0.3))

optimizer = create_optimizer(final_model, best_p.get('optimizer', 'Adam'), best_p['learning_rate'])
scheduler = create_scheduler(optimizer)
final_trainer = Trainer(final_model, nn.CrossEntropyLoss(), optimizer, device, scheduler)

final_history = final_trainer.fit(train_loader, val_loader, num_epochs=25,
                                  early_stopping_patience=7, model_name='custom_cnn_best')
print(f"Final Accuracy: {max(final_history['val_acc']):.4f}")

## 10. ONNX Export

In [ ]:
from src.export.onnx_export import export_to_onnx, verify_onnx_model

trainer.load_best_model('custom_cnn')
export_to_onnx(custom_model, 'models/custom_cnn.onnx')
verify_onnx_model('models/custom_cnn.onnx')

trainer_resnet.load_best_model('resnet18')
export_to_onnx(resnet_model, 'models/resnet18.onnx')
verify_onnx_model('models/resnet18.onnx')

## 11. Dosyaları İndir

In [ ]:
from google.colab import files

for f in ['models/custom_cnn_best.pt', 'models/resnet18_best.pt',
          'models/custom_cnn.onnx', 'models/resnet18.onnx',
          'models/results.json']:
    if os.path.exists(f):
        files.download(f)
        print(f'  ✅ {f}')
    else:
        print(f'  ⚠️ {f} bulunamadı')

print('\n🎉 Dosyaları lokal projendeki models/ klasörüne kopyala.')
print('Sonra: streamlit run app/app.py')